# Sunspot Data — Exploratory Data Analysis

Before modelling, we explore the SILSO daily sunspot dataset to understand its structure and justify our modelling choices.

**Why this matters:**  
Solar activity directly affects satellite operations, GPS accuracy, power grids, and HF radio communications. Geomagnetic storms triggered by high sunspot activity can cause billion-dollar damage to infrastructure. Accurate short-term forecasts help operators take preventive action.

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Robust project root detection — works regardless of kernel working directory
_root = os.getcwd()
for _ in range(4):
    if os.path.exists(os.path.join(_root, 'config.yaml')):
        break
    _root = os.path.dirname(_root)
os.chdir(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)

os.makedirs('reports', exist_ok=True)

from src.utils import load_config
from src.data import load_data
from src.features import create_features, prepare_target
from src.train import expanding_walk_forward_splits

config = load_config('config.yaml')
df = load_data(config['data']['url'], save_path=config['data']['save_path'])

start_period = '1965-01-01'
df_filtered = df.loc[start_period:].copy()
print(f"Working directory: {os.getcwd()}")
print(f"Period: {df_filtered.index.min().date()} → {df_filtered.index.max().date()}")
print(f"Total days: {len(df_filtered):,}")

## 1. Full Time Series

In [ ]:
fig, ax = plt.subplots(figsize=(15, 4))
ax.plot(df_filtered.index, df_filtered['SUNSPOTS'], lw=0.5, color='steelblue', alpha=0.7)
ax.plot(df_filtered.index, df_filtered['SUNSPOTS'].rolling(365).mean(),
        color='red', lw=1.8, label='365-day rolling mean')
ax.set_title('Daily International Sunspot Number (1965–present)', fontsize=13)
ax.set_xlabel('Year')
ax.set_ylabel('Sunspot count')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
fig.savefig('reports/01_time_series.png', dpi=150, bbox_inches='tight')
plt.show()

zero_days = (df_filtered['SUNSPOTS'] == 0).sum()
print(f"Max:  {df_filtered['SUNSPOTS'].max():.0f}")
print(f"Mean: {df_filtered['SUNSPOTS'].mean():.1f}")
print(f"Std:  {df_filtered['SUNSPOTS'].std():.1f}")
print(f"Days with 0 sunspots: {zero_days:,} ({zero_days/len(df_filtered)*100:.1f}%) — common near solar minimum")

## 2. Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(df_filtered['SUNSPOTS'], bins=60, color='steelblue', edgecolor='white')
axes[0].axvline(df_filtered['SUNSPOTS'].mean(), color='red', linestyle='--',
                label=f"Mean: {df_filtered['SUNSPOTS'].mean():.1f}")
axes[0].set_title('Raw Sunspot Count')
axes[0].set_xlabel('Sunspots')
axes[0].set_ylabel('Frequency')
axes[0].legend()

axes[1].hist(df_filtered['LOG_SUNSPOTS'], bins=60, color='darkorange', edgecolor='white')
axes[1].set_title('log(1+x) Transformed')
axes[1].set_xlabel('log(1 + Sunspots)')
axes[1].set_ylabel('Frequency')

plt.suptitle('Log transform stabilises the heavy-tailed distribution → justified for modelling', y=1.02)
plt.tight_layout()
fig.savefig('reports/02_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Autocorrelation & Stationarity

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(df_filtered['SUNSPOTS'].dropna(), lags=60, ax=axes[0], color='steelblue')
axes[0].set_title('Autocorrelation (ACF) — first 60 lags')
plot_pacf(df_filtered['SUNSPOTS'].dropna(), lags=60, ax=axes[1], color='darkorange', method='ywm')
axes[1].set_title('Partial Autocorrelation (PACF)')
plt.tight_layout()
fig.savefig('reports/03_acf_pacf.png', dpi=150, bbox_inches='tight')
plt.show()

adf_stat, adf_p, *_ = adfuller(df_filtered['SUNSPOTS'].dropna())
print(f"ADF statistic: {adf_stat:.4f}  |  p-value: {adf_p:.4f}")
print("→ Series is stationary (p < 0.05)" if adf_p < 0.05 else "→ Series is NOT stationary")
print()
print("Key insight: strong short-lag autocorrelation justifies using lag features (lag-1 to lag-7).")
print("Slow decay suggests the 11-year cycle structure — justifies the lag-365 and lag-4015 features.")

## 4. Solar Cycle (~11 years)

In [ ]:
cycle_df = df_filtered.loc['1996-01-01':'2019-12-31']
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(cycle_df.index, cycle_df['SUNSPOTS'], lw=0.6, color='steelblue', alpha=0.7)
ax.plot(cycle_df.index, cycle_df['SUNSPOTS'].rolling(365).mean(),
        color='red', lw=2, label='365-day rolling mean')
for span in [('1996-01-01', '1997-06-01'), ('2008-01-01', '2009-12-31'), ('2019-01-01', '2019-12-31')]:
    ax.axvspan(*span, alpha=0.08, color='gray')
ax.text(pd.Timestamp('1996-06-01'), 5, 'Min', fontsize=9, color='gray')
ax.text(pd.Timestamp('2008-06-01'), 5, 'Min', fontsize=9, color='gray')
ax.text(pd.Timestamp('2019-04-01'), 5, 'Min', fontsize=9, color='gray')
ax.set_title('Two complete solar cycles (1996–2019) — ~11 years each', fontsize=13)
ax.set_xlabel('Year')
ax.set_ylabel('Sunspot count')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
fig.savefig('reports/04_solar_cycles.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Naive Baselines

Before training any model, we establish naive baselines using the same walk-forward framework.  
The hybrid model is compared against these in `01-Analysis_and_Modeling.ipynb`.

- **Naive (lag-1):** predict tomorrow = today's value
- **Rolling mean (30d):** average of last 30 days
- **Rolling mean (365d):** average of last year — stronger baseline used in solar forecasting literature


In [ ]:
df_feat = create_features(df_filtered, config['features']['lags'], config['features']['rolling_windows'])
df_feat = prepare_target(df_feat, shift=config['features']['target_shift'])
X = df_feat.drop(columns=['target', 'SUNSPOTS', 'LOG_SUNSPOTS'])
y = df_feat['target']

naive_rmse, naive_mae   = [], []
roll30_rmse, roll30_mae = [], []
roll365_rmse, roll365_mae = [], []

for Xtr, ytr, Xval, yval in expanding_walk_forward_splits(
    X, y,
    config['evaluation']['initial_train_size'],
    config['evaluation']['val_size'],
    config['evaluation']['step_size']
):
    if len(Xtr) < 365:
        continue
    yval_linear  = np.expm1(yval)
    naive_pred   = np.full(len(yval), np.expm1(ytr.iloc[-1]))
    roll30_pred  = np.full(len(yval), np.expm1(ytr.iloc[-30:].mean()))
    roll365_pred = np.full(len(yval), np.expm1(ytr.iloc[-365:].mean()))

    naive_rmse.append(np.sqrt(mean_squared_error(yval_linear, naive_pred)))
    naive_mae.append(mean_absolute_error(yval_linear, naive_pred))
    roll30_rmse.append(np.sqrt(mean_squared_error(yval_linear, roll30_pred)))
    roll30_mae.append(mean_absolute_error(yval_linear, roll30_pred))
    roll365_rmse.append(np.sqrt(mean_squared_error(yval_linear, roll365_pred)))
    roll365_mae.append(mean_absolute_error(yval_linear, roll365_pred))

print(f"{'Baseline':<24} {'RMSE':>8} {'MAE':>8}")
print("─" * 42)
print(f"{'Naive (lag-1)':<24} {np.mean(naive_rmse):>8.2f} {np.mean(naive_mae):>8.2f}")
print(f"{'Rolling mean (30d)':<24} {np.mean(roll30_rmse):>8.2f} {np.mean(roll30_mae):>8.2f}")
print(f"{'Rolling mean (365d)':<24} {np.mean(roll365_rmse):>8.2f} {np.mean(roll365_mae):>8.2f}")
print()
print("→ These are the targets to beat. See 01-Analysis_and_Modeling.ipynb for the full comparison.")
